# Load data using PySpark

## Load a csv file into a dataframe

In [0]:
df_orders = spark.read.options(header=True, inferSchema=True).csv('/Workspace/Users/kathaszynka@gmail.com/olist_project/data/raw/olist_orders_dataset.csv')

In [0]:
df_orders.display()

In [0]:
df_orders.printSchema()

## Leave only the necessary columns

In [0]:
df_orders_selected = df_orders.select(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date'])

df_orders_selected.display()

# Delivery status analysis

In [0]:
# Select only the delivered orders
df_delivered = df_orders_selected.filter(df_orders_selected.order_status == "delivered")

df_delivered.display()

In [0]:
# Print the number of delivered orders
print(f"The number of delivered orders: {df_delivered.count()}")

In [0]:

from pyspark.sql import functions as F

# Add column is_late
# 1 - the package was late
# 0 - the package was delivered on time

df_delivered = df_delivered.withColumn('is_late', F.when(df_delivered.order_delivered_customer_date > df_delivered.order_estimated_delivery_date, 1).otherwise(0))

df_delivered.display()

In [0]:
# display the late orders
df_delivered.filter(df_delivered.is_late == 1).display()

In [0]:
# Count the delivery delay days

df_delivered = df_delivered.withColumn('delivery_delay_days', Fdatediff('order_delivered_customer_date', 'order_estimated_delivery_date'))

In [0]:
# display the late orders
df_delivered.filter(df_delivered.is_late == 1).display()

# Aggregations

In [0]:
df_orders.groupBy('order_status').count().sort('count', ascending=False).display()

# Nulls

In [0]:
# this would remove all rows where theres at least one null column
df_orders.na.drop().display()

In [0]:
# removes a row only if ALL columns ale null
df_orders.na.drop(how='all').display()

In [0]:
# drop rows that have less than 2 non-null and non-nan values
df_orders.na.drop(thresh=2).display()

In [0]:
# how -> all or any (all: drop row if all columns are null; any: drop row if ANY of the columns is null)

# tresh: drop rows that have less than tresh non-null and non-nan values


# subset: drop orws with null and naN values in the specified columns




In [0]:
# count how many null values per column
from pyspark.sql import functions as F

df_orders.select(
    F.count(
        F.when(F.col("order_delivered_customer_date").isNull(), 1)
    ).alias("null_count")
).display()

In [0]:
df_orders.select(
    F.count(
        F.when(F.col("order_delivered_customer_date").isNull(), 1)
    )
).display()

In [0]:
null_counts = [
    F.count(
        F.when(F.col(c).isNull(), 1)     
    ).alias(c)
    for c in df_orders.columns
]

df_orders.select(*null_counts).display()

# Join Orders with Customers
## Load customers data

In [0]:
df_customers = spark.read.options(header=True, inferSchema=True).csv('/Workspace/Users/kathaszynka@gmail.com/olist_project/data/raw/olist_customers_dataset.csv')

df_customers.display()

In [0]:
df_customers.printSchema()

## Join Orders with Customers

In [0]:
df_orders_customers = df_orders.join(df_customers, "customer_id", how='left')

# More aggregations

In [0]:
df_orders_customers.groupBy('customer_state').count().sort('count', ascending=False).show()

In [0]:
df_orders.agg(
    F.count("order_id").alias("total_orders"),
    F.count_distinct("customer_id").alias("unique_customers"),
    F.min("order_purchase_timestamp").alias("first_order"),
    F.max("order_purchase_timestamp").alias("last_order")
).show()